In [1]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import re
import textwrap
import html as html_lib

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
CSV = r"C:\Users\suzan\Downloads\MOP\MOP_mega_with_locations_fixed.csv"
OUT_FILE = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html")

HIGHLIGHT_YEAR = "2025"
COLOR_DEFAULT = "#9CA3AF"   # gray
COLOR_2025 = "#2563EB"      # highlight color

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def slugify(s: str, max_len: int = 80) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\s\-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = s.strip("-")
    if not s:
        s = "unknown"
    return s[:max_len]

def to_int_year(y):
    try:
        return int(str(y))
    except Exception:
        return None

def fmt_lbs(x):
    try:
        return f"{float(x):,.0f} lbs"
    except Exception:
        return f"{x} lbs"

def esc(s: str) -> str:
    return html_lib.escape(str(s))

# ------------------------------------------------------------------
# Load & normalize
# ------------------------------------------------------------------
df = pd.read_csv(CSV)

name_col = (
    "Restaurant"
    if "Restaurant" in df.columns
    else ("Restaurants" if "Restaurants" in df.columns else ("name" if "name" in df.columns else None))
)
if name_col is None:
    raise ValueError("Could not find Restaurant column (Restaurant/Restaurants/name).")

weight_col = next((c for c in df.columns if "shell" in c.lower() and "weight" in c.lower()), None)
if not weight_col:
    raise ValueError("No 'Shell Only Weight' column found (shell + weight).")

# Ensure Year exists
if "Year" not in df.columns:
    date_like = next((c for c in df.columns if "date" in c.lower()), None)
    if not date_like:
        raise ValueError("No Year column and no date-like column found to derive Year.")
    df["Year"] = pd.to_datetime(df[date_like], errors="coerce").dt.year

df["Year"] = df["Year"].astype("Int64").astype(str).replace("<NA>", "Unknown")
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)

# Keep only positive weights for impact reporting
df_pos = df[df[weight_col] > 0].copy()

restaurants = sorted(df_pos[name_col].fillna("Unknown").astype(str).unique())
if not restaurants:
    raise ValueError("No restaurants with Shell Only Weight > 0 found.")

print(f"Found {len(restaurants)} restaurants with >0 weight. Writing one report to: {OUT_FILE}")

# ------------------------------------------------------------------
# Build per-restaurant sections
# ------------------------------------------------------------------
toc_items = []
sections_html = []

for r in restaurants:
    sub = df_pos[df_pos[name_col].astype(str) == str(r)].copy()

    year_totals = (
        sub.groupby("Year", as_index=False)[weight_col]
        .sum()
        .rename(columns={weight_col: "Weight"})
    )

    year_totals["YearInt"] = year_totals["Year"].apply(to_int_year)
    year_totals = year_totals.sort_values(["YearInt", "Year"], na_position="last")

    years = year_totals["Year"].tolist()
    weights = year_totals["Weight"].tolist()

    total_all = float(year_totals["Weight"].sum())
    w_2025 = float(year_totals.loc[year_totals["Year"] == HIGHLIGHT_YEAR, "Weight"].sum())

    best_idx = int(year_totals["Weight"].idxmax())
    best_year = str(year_totals.loc[best_idx, "Year"])
    best_val = float(year_totals.loc[best_idx, "Weight"])
    avg_val = total_all / max(len(year_totals), 1)

    if HIGHLIGHT_YEAR in set(years):
        pct_total = (w_2025 / total_all) * 100 if total_all > 0 else 0
        delta_vs_avg = w_2025 - avg_val
        delta_word = "above" if delta_vs_avg >= 0 else "below"
        delta_abs = abs(delta_vs_avg)
        summary = (
            f"In {HIGHLIGHT_YEAR}, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"That represents <b>{pct_total:.1f}%</b> of this restaurant’s recorded contributions across all years. "
            f"Compared with this restaurant’s average year (<b>{fmt_lbs(avg_val)}</b>), {HIGHLIGHT_YEAR} was "
            f"<b>{fmt_lbs(delta_abs)} {delta_word}</b> average. "
            f"The highest recorded year was <b>{esc(best_year)}</b> with <b>{fmt_lbs(best_val)}</b>."
        )
        badge = f'<span class="badge badge-yes">{HIGHLIGHT_YEAR} data</span>'
    else:
        summary = (
            f"<b>{esc(r)}</b> has recorded shell recycling contributions in {len(year_totals)} year(s), "
            f"totaling <b>{fmt_lbs(total_all)}</b>. "
            f"No contributions were recorded for <b>{HIGHLIGHT_YEAR}</b> in this dataset. "
            f"The highest recorded year was <b>{esc(best_year)}</b> with <b>{fmt_lbs(best_val)}</b>."
        )
        badge = f'<span class="badge badge-no">No {HIGHLIGHT_YEAR}</span>'

    colors = [COLOR_2025 if str(y) == HIGHLIGHT_YEAR else COLOR_DEFAULT for y in years]

    fig = go.Figure(
        data=[
            go.Bar(
                x=years,
                y=weights,
                marker=dict(color=colors),
                hovertemplate="<b>Year</b>: %{x}<br><b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
            )
        ]
    )

    fig.update_layout(
        title=f"{r} — Shell Recycling Impact (Yearly Totals)",
        xaxis=dict(title="Year", type="category"),
        yaxis=dict(title="Pounds diverted (Shell Only Weight)", rangemode="tozero"),
        margin=dict(l=60, r=30, t=60, b=45),
        height=360,
    )

    # IMPORTANT: full_html=False so we can embed many charts; Plotly JS loaded once at top
    chart_div = fig.to_html(include_plotlyjs=False, full_html=False, config={"displaylogo": False})

    anchor = slugify(r)
    toc_items.append(
        f'<li><a href="#{anchor}">{esc(r)}</a> {badge}</li>'
    )

    sections_html.append(
        textwrap.dedent(
            f"""
            <section class="report-card" id="{anchor}">
              <div class="report-head">
                <h2>{esc(r)}</h2>
                <div class="report-meta">
                  <span class="pill">Total (all years): <b>{fmt_lbs(total_all)}</b></span>
                  <span class="pill pill-2025">{HIGHLIGHT_YEAR}: <b>{fmt_lbs(w_2025) if HIGHLIGHT_YEAR in set(years) else "0 lbs"}</b></span>
                </div>
              </div>
              <p class="summary">{summary}</p>
              <div class="chart-wrap">{chart_div}</div>
              <div class="backtotop"><a href="#top">Back to top</a></div>
            </section>
            """
        ).strip()
    )

# ------------------------------------------------------------------
# Assemble one HTML document
# ------------------------------------------------------------------
toc_html = "\n".join(toc_items)
body_sections = "\n\n".join(sections_html)

html_doc = textwrap.dedent(
    f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>2025 Shell Recycling Impact Reports — All Restaurants</title>

  <!-- Load Plotly ONCE -->
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>

  <style>
    :root {{
      --bg: #ffffff;
      --muted: #6b7280;
      --card: #f8fafc;
      --border: #e5e7eb;
      --accent: {COLOR_2025};
    }}

    body {{
      margin: 0;
      font-family: Arial, Helvetica, sans-serif;
      background: var(--bg);
      color: #111;
    }}

    .container {{
      max-width: 1100px;
      margin: 0 auto;
      padding: 22px 18px 60px;
    }}

    h1 {{
      font-size: 24px;
      margin: 0 0 6px;
    }}

    .subtitle {{
      color: var(--muted);
      font-size: 13px;
      margin: 0 0 16px;
      line-height: 1.4;
    }}

    .toc {{
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 14px 16px;
      margin: 14px 0 18px;
    }}

    .toc h3 {{
      margin: 0 0 10px;
      font-size: 15px;
    }}

    .toc ul {{
      margin: 0;
      padding-left: 18px;
      columns: 2;
      column-gap: 26px;
    }}

    @media (max-width: 900px) {{
      .toc ul {{ columns: 1; }}
    }}

    .toc li {{
      margin: 6px 0;
      break-inside: avoid;
    }}

    .toc a {{
      color: #111;
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .toc a:hover {{
      border-bottom-style: solid;
    }}

    .badge {{
      display: inline-block;
      margin-left: 8px;
      font-size: 11px;
      padding: 2px 8px;
      border-radius: 999px;
      border: 1px solid var(--border);
      vertical-align: middle;
      white-space: nowrap;
    }}
    .badge-yes {{
      background: rgba(37, 99, 235, 0.10);
      border-color: rgba(37, 99, 235, 0.35);
      color: #1d4ed8;
    }}
    .badge-no {{
      background: rgba(107, 114, 128, 0.10);
      border-color: rgba(107, 114, 128, 0.35);
      color: #374151;
    }}

    .report-card {{
      border: 1px solid var(--border);
      border-radius: 14px;
      padding: 14px 16px 10px;
      margin: 14px 0;
      background: #fff;
    }}

    .report-head {{
      display: flex;
      align-items: flex-start;
      justify-content: space-between;
      gap: 12px;
      flex-wrap: wrap;
    }}

    .report-card h2 {{
      font-size: 18px;
      margin: 0;
    }}

    .report-meta {{
      display: flex;
      gap: 8px;
      flex-wrap: wrap;
      justify-content: flex-end;
    }}

    .pill {{
      font-size: 12px;
      color: #111;
      background: var(--card);
      border: 1px solid var(--border);
      padding: 4px 10px;
      border-radius: 999px;
    }}

    .pill-2025 {{
      border-color: rgba(37, 99, 235, 0.45);
      background: rgba(37, 99, 235, 0.08);
    }}

    .summary {{
      margin: 10px 0 10px;
      color: #111;
      line-height: 1.45;
    }}

    .chart-wrap {{
      margin-top: 6px;
    }}

    .backtotop {{
      margin-top: 8px;
      font-size: 12px;
    }}
    .backtotop a {{
      color: var(--muted);
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .backtotop a:hover {{
      border-bottom-style: solid;
    }}

    .footer-note {{
      margin-top: 18px;
      color: var(--muted);
      font-size: 12px;
      line-height: 1.4;
    }}
  </style>
</head>

<body>
  <div class="container" id="top">
    <h1>2025 Shell Recycling Impact Reports — All Restaurants</h1>
    <p class="subtitle">
      Metric: <b>Shell Only Weight (lbs)</b> • 2025 is highlighted in <span style="color:var(--accent);font-weight:bold;">blue</span>.
      This page aggregates each restaurant’s recorded contributions by year where Shell Only Weight &gt; 0.
    </p>

    <div class="toc">
      <h3>Restaurant index</h3>
      <ul>
        {toc_html}
      </ul>
    </div>

    {body_sections}

    <div class="footer-note">
      Note: Events without coordinates are not region-attributed here unless they are labeled as the restaurant itself.
      Source: MOP_mega_with_locations_fixed.csv
    </div>
  </div>
</body>
</html>
"""
).strip()

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(html_doc, encoding="utf-8", errors="replace")
print(f"Saved: {OUT_FILE}")


Found 35 restaurants with >0 weight. Writing one report to: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Saved: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html


In [3]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import re
import textwrap
import html as html_lib

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
CSV = r"C:\Users\suzan\Downloads\MOP\MOP_mega_with_locations_fixed.csv"
OUT_FILE = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html")

HIGHLIGHT_YEAR = "2025"
COLOR_DEFAULT = "#9CA3AF"   # gray
COLOR_2025 = "#2563EB"      # highlight color

# Layout tuning (to avoid thick boxes)
CHART_HEIGHT = 280          # was ~360; tighter
CARD_PADDING = "10px 12px"  # was bigger
CARD_MARGIN = "10px 0"      # smaller vertical spacing

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def slugify(s: str, max_len: int = 80) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\s\-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = s.strip("-")
    if not s:
        s = "unknown"
    return s[:max_len]

def to_int_year(y):
    try:
        return int(str(y))
    except Exception:
        return None

def fmt_lbs(x):
    try:
        return f"{float(x):,.0f} lbs"
    except Exception:
        return f"{x} lbs"

def esc(s: str) -> str:
    return html_lib.escape(str(s))

# ------------------------------------------------------------------
# Load & normalize
# ------------------------------------------------------------------
df = pd.read_csv(CSV)

name_col = (
    "Restaurant"
    if "Restaurant" in df.columns
    else ("Restaurants" if "Restaurants" in df.columns else ("name" if "name" in df.columns else None))
)
if name_col is None:
    raise ValueError("Could not find Restaurant column (Restaurant/Restaurants/name).")

weight_col = next((c for c in df.columns if "shell" in c.lower() and "weight" in c.lower()), None)
if not weight_col:
    raise ValueError("No 'Shell Only Weight' column found (shell + weight).")

# Ensure Year exists
if "Year" not in df.columns:
    date_like = next((c for c in df.columns if "date" in c.lower()), None)
    if not date_like:
        raise ValueError("No Year column and no date-like column found to derive Year.")
    df["Year"] = pd.to_datetime(df[date_like], errors="coerce").dt.year

df["Year"] = df["Year"].astype("Int64").astype(str).replace("<NA>", "Unknown")
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)

# Keep only positive weights for impact reporting
df_pos = df[df[weight_col] > 0].copy()

# ------------------------------------------------------------------
# NEW: only restaurants active in 2025
# ------------------------------------------------------------------
df_2025 = df_pos[df_pos["Year"].astype(str) == HIGHLIGHT_YEAR].copy()
restaurants_2025 = sorted(df_2025[name_col].fillna("Unknown").astype(str).unique())

if not restaurants_2025:
    raise ValueError(f"No restaurants with Shell Only Weight > 0 found for {HIGHLIGHT_YEAR}.")

print(f"Restaurants active in {HIGHLIGHT_YEAR}: {len(restaurants_2025)}")
print(f"Writing one report to: {OUT_FILE}")

# ------------------------------------------------------------------
# Build per-restaurant sections
# ------------------------------------------------------------------
toc_items = []
sections_html = []

for r in restaurants_2025:
    sub = df_pos[df_pos[name_col].astype(str) == str(r)].copy()

    year_totals = (
        sub.groupby("Year", as_index=False)[weight_col]
        .sum()
        .rename(columns={weight_col: "Weight"})
    )

    year_totals["YearInt"] = year_totals["Year"].apply(to_int_year)
    year_totals = year_totals.sort_values(["YearInt", "Year"], na_position="last")

    years = year_totals["Year"].tolist()
    weights = year_totals["Weight"].tolist()

    total_all = float(year_totals["Weight"].sum())
    w_2025 = float(year_totals.loc[year_totals["Year"] == HIGHLIGHT_YEAR, "Weight"].sum())

    best_idx = int(year_totals["Weight"].idxmax())
    best_year = str(year_totals.loc[best_idx, "Year"])
    best_val = float(year_totals.loc[best_idx, "Weight"])
    avg_val = total_all / max(len(year_totals), 1)

    # Since we filtered to 2025-active restaurants, 2025 should be present,
    # but keep robust logic anyway.
    if HIGHLIGHT_YEAR in set(years):
        pct_total = (w_2025 / total_all) * 100 if total_all > 0 else 0
        delta_vs_avg = w_2025 - avg_val
        delta_word = "above" if delta_vs_avg >= 0 else "below"
        delta_abs = abs(delta_vs_avg)

        # Slightly cleaner narrative for the “single-year” case
        if len(year_totals) == 1 and best_year == HIGHLIGHT_YEAR:
            summary = (
                f"In {HIGHLIGHT_YEAR}, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
                f"This restaurant has recorded contributions for <b>1 year</b> in the current dataset."
            )
        else:
            summary = (
                f"In {HIGHLIGHT_YEAR}, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
                f"That represents <b>{pct_total:.1f}%</b> of this restaurant’s recorded contributions across all years. "
                f"Compared with this restaurant’s average year (<b>{fmt_lbs(avg_val)}</b>), {HIGHLIGHT_YEAR} was "
                f"<b>{fmt_lbs(delta_abs)} {delta_word}</b> average. "
                f"The highest recorded year was <b>{esc(best_year)}</b> with <b>{fmt_lbs(best_val)}</b>."
            )
        badge = f'<span class="badge badge-yes">{HIGHLIGHT_YEAR} active</span>'
    else:
        # This should not occur given restaurants_2025 filter, but keep safe.
        summary = (
            f"<b>{esc(r)}</b> has recorded shell recycling contributions totaling <b>{fmt_lbs(total_all)}</b>, "
            f"but no {HIGHLIGHT_YEAR} entry was found after filtering."
        )
        badge = f'<span class="badge badge-no">Missing {HIGHLIGHT_YEAR}</span>'

    colors = [COLOR_2025 if str(y) == HIGHLIGHT_YEAR else COLOR_DEFAULT for y in years]

    fig = go.Figure(
        data=[
            go.Bar(
                x=years,
                y=weights,
                marker=dict(color=colors),
                hovertemplate="<b>Year</b>: %{x}<br><b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
            )
        ]
    )

    fig.update_layout(
        title=f"{r} — Yearly Totals",
        xaxis=dict(title="Year", type="category"),
        yaxis=dict(title="Pounds diverted (Shell Only Weight)", rangemode="tozero"),
        margin=dict(l=60, r=24, t=50, b=42),
        height=CHART_HEIGHT,
    )

    chart_div = fig.to_html(include_plotlyjs=False, full_html=False, config={"displaylogo": False})

    anchor = slugify(r)
    toc_items.append(f'<li><a href="#{anchor}">{esc(r)}</a> {badge}</li>')

    sections_html.append(
        textwrap.dedent(
            f"""
            <section class="report-card" id="{anchor}">
              <div class="report-head">
                <h2>{esc(r)}</h2>
                <div class="report-meta">
                  <span class="pill pill-2025">{HIGHLIGHT_YEAR}: <b>{fmt_lbs(w_2025)}</b></span>
                  <span class="pill">All years: <b>{fmt_lbs(total_all)}</b></span>
                </div>
              </div>
              <p class="summary">{summary}</p>
              <div class="chart-wrap">{chart_div}</div>
              <div class="backtotop"><a href="#top">Back to top</a></div>
            </section>
            """
        ).strip()
    )

# ------------------------------------------------------------------
# Assemble one HTML document
# ------------------------------------------------------------------
toc_html = "\n".join(toc_items)
body_sections = "\n\n".join(sections_html)

html_doc = textwrap.dedent(
    f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Active Restaurants</title>

  <!-- Load Plotly ONCE -->
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>

  <style>
    :root {{
      --bg: #ffffff;
      --muted: #6b7280;
      --card: #f8fafc;
      --border: #e5e7eb;
      --accent: {COLOR_2025};
    }}

    body {{
      margin: 0;
      font-family: Arial, Helvetica, sans-serif;
      background: var(--bg);
      color: #111;
    }}

    .container {{
      max-width: 1100px;
      margin: 0 auto;
      padding: 18px 16px 50px;
    }}

    h1 {{
      font-size: 22px;
      margin: 0 0 6px;
    }}

    .subtitle {{
      color: var(--muted);
      font-size: 13px;
      margin: 0 0 12px;
      line-height: 1.35;
    }}

    .toc {{
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 10px 12px;
      margin: 10px 0 14px;
    }}

    .toc h3 {{
      margin: 0 0 8px;
      font-size: 14px;
    }}

    .toc ul {{
      margin: 0;
      padding-left: 18px;
      columns: 2;
      column-gap: 22px;
    }}

    @media (max-width: 900px) {{
      .toc ul {{ columns: 1; }}
    }}

    .toc li {{
      margin: 5px 0;
      break-inside: avoid;
    }}

    .toc a {{
      color: #111;
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .toc a:hover {{
      border-bottom-style: solid;
    }}

    .badge {{
      display: inline-block;
      margin-left: 8px;
      font-size: 11px;
      padding: 2px 8px;
      border-radius: 999px;
      border: 1px solid var(--border);
      vertical-align: middle;
      white-space: nowrap;
    }}
    .badge-yes {{
      background: rgba(37, 99, 235, 0.10);
      border-color: rgba(37, 99, 235, 0.35);
      color: #1d4ed8;
    }}
    .badge-no {{
      background: rgba(107, 114, 128, 0.10);
      border-color: rgba(107, 114, 128, 0.35);
      color: #374151;
    }}

    /* Thinner cards */
    .report-card {{
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: {CARD_PADDING};
      margin: {CARD_MARGIN};
      background: #fff;
    }}

    .report-head {{
      display: flex;
      align-items: flex-start;
      justify-content: space-between;
      gap: 10px;
      flex-wrap: wrap;
    }}

    .report-card h2 {{
      font-size: 16px;
      margin: 0;
    }}

    .report-meta {{
      display: flex;
      gap: 6px;
      flex-wrap: wrap;
      justify-content: flex-end;
    }}

    .pill {{
      font-size: 12px;
      color: #111;
      background: var(--card);
      border: 1px solid var(--border);
      padding: 3px 9px;
      border-radius: 999px;
    }}

    .pill-2025 {{
      border-color: rgba(37, 99, 235, 0.45);
      background: rgba(37, 99, 235, 0.08);
    }}

    .summary {{
      margin: 8px 0 8px;
      color: #111;
      line-height: 1.35;
      font-size: 13px;
    }}

    .chart-wrap {{
      margin-top: 4px;
    }}

    .backtotop {{
      margin-top: 6px;
      font-size: 12px;
    }}
    .backtotop a {{
      color: var(--muted);
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .backtotop a:hover {{
      border-bottom-style: solid;
    }}

    .footer-note {{
      margin-top: 14px;
      color: var(--muted);
      font-size: 12px;
      line-height: 1.35;
    }}
  </style>
</head>

<body>
  <div class="container" id="top">
    <h1>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Restaurants Active in {HIGHLIGHT_YEAR}</h1>
    <p class="subtitle">
      Metric: <b>Shell Only Weight (lbs)</b> • Only restaurants with recorded contributions in <b>{HIGHLIGHT_YEAR}</b> are included.
      {HIGHLIGHT_YEAR} is highlighted in <span style="color:var(--accent);font-weight:bold;">blue</span>.
    </p>

    <div class="toc">
      <h3>Restaurant index ({len(restaurants_2025)})</h3>
      <ul>
        {toc_html}
      </ul>
    </div>

    {body_sections}

    <div class="footer-note">
      Note: This report aggregates each restaurant’s contributions by year where Shell Only Weight &gt; 0.
      Source: MOP_mega_with_locations_fixed.csv
    </div>
  </div>
</body>
</html>
"""
).strip()

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(html_doc, encoding="utf-8", errors="replace")
print(f"Saved: {OUT_FILE}")

Restaurants active in 2025: 25
Writing one report to: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Saved: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html


In [5]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import re
import textwrap
import html as html_lib

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
CSV = r"C:\Users\suzan\Downloads\MOP\MOP_mega_with_locations_fixed.csv"
OUT_FILE = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html")

HIGHLIGHT_YEAR = "2025"
COLOR_DEFAULT = "#9CA3AF"   # gray
COLOR_2025 = "#2563EB"      # highlight color

# Layout tuning (thin cards)
CHART_HEIGHT = 280
CARD_PADDING = "10px 12px"
CARD_MARGIN = "10px 0"

# Plot spacing consistency
BAR_WIDTH = 0.5     # keeps bars from getting “fat” when few categories
BARGAP = 0.35       # consistent spacing between bars

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def slugify(s: str, max_len: int = 80) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\s\-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = s.strip("-")
    if not s:
        s = "unknown"
    return s[:max_len]

def to_int_year(y):
    try:
        return int(str(y))
    except Exception:
        return None

def fmt_lbs(x):
    try:
        return f"{float(x):,.0f} lbs"
    except Exception:
        return f"{x} lbs"

def esc(s: str) -> str:
    return html_lib.escape(str(s))

# ------------------------------------------------------------------
# Load & normalize
# ------------------------------------------------------------------
df = pd.read_csv(CSV)

name_col = (
    "Restaurant"
    if "Restaurant" in df.columns
    else ("Restaurants" if "Restaurants" in df.columns else ("name" if "name" in df.columns else None))
)
if name_col is None:
    raise ValueError("Could not find Restaurant column (Restaurant/Restaurants/name).")

weight_col = next((c for c in df.columns if "shell" in c.lower() and "weight" in c.lower()), None)
if not weight_col:
    raise ValueError("No 'Shell Only Weight' column found (shell + weight).")

# Ensure Year exists
if "Year" not in df.columns:
    date_like = next((c for c in df.columns if "date" in c.lower()), None)
    if not date_like:
        raise ValueError("No Year column and no date-like column found to derive Year.")
    df["Year"] = pd.to_datetime(df[date_like], errors="coerce").dt.year

df["Year"] = df["Year"].astype("Int64").astype(str).replace("<NA>", "Unknown")
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)

# Keep only positive weights for impact reporting
df_pos = df[df[weight_col] > 0].copy()

# ------------------------------------------------------------------
# Only restaurants active in 2025
# ------------------------------------------------------------------
df_2025 = df_pos[df_pos["Year"].astype(str) == HIGHLIGHT_YEAR].copy()
restaurants_2025 = sorted(df_2025[name_col].fillna("Unknown").astype(str).unique())

if not restaurants_2025:
    raise ValueError(f"No restaurants with Shell Only Weight > 0 found for {HIGHLIGHT_YEAR}.")

print(f"Restaurants active in {HIGHLIGHT_YEAR}: {len(restaurants_2025)}")
print(f"Writing one report to: {OUT_FILE}")

# ------------------------------------------------------------------
# Build per-restaurant sections
# ------------------------------------------------------------------
toc_items = []
sections_html = []

for r in restaurants_2025:
    sub = df_pos[df_pos[name_col].astype(str) == str(r)].copy()

    year_totals = (
        sub.groupby("Year", as_index=False)[weight_col]
        .sum()
        .rename(columns={weight_col: "Weight"})
    )

    # Sort years numerically when possible
    year_totals["YearInt"] = year_totals["Year"].apply(to_int_year)
    year_totals = year_totals.sort_values(["YearInt", "Year"], na_position="last")

    years = year_totals["Year"].tolist()
    weights = year_totals["Weight"].tolist()

    total_all = float(year_totals["Weight"].sum())
    w_2025 = float(year_totals.loc[year_totals["Year"] == HIGHLIGHT_YEAR, "Weight"].sum())

    best_idx = int(year_totals["Weight"].idxmax())
    best_year = str(year_totals.loc[best_idx, "Year"])
    best_val = float(year_totals.loc[best_idx, "Weight"])
    avg_val = total_all / max(len(year_totals), 1)

    # Narrative (cleaner for single-year restaurants)
    if len(year_totals) == 1 and best_year == HIGHLIGHT_YEAR:
        summary = (
            f"In {HIGHLIGHT_YEAR}, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"This restaurant has recorded contributions for <b>1 year</b> in the current dataset."
        )
    else:
        pct_total = (w_2025 / total_all) * 100 if total_all > 0 else 0
        delta_vs_avg = w_2025 - avg_val
        delta_word = "above" if delta_vs_avg >= 0 else "below"
        delta_abs = abs(delta_vs_avg)
        summary = (
            f"In {HIGHLIGHT_YEAR}, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"That represents <b>{pct_total:.1f}%</b> of this restaurant’s recorded contributions across all years. "
            f"Compared with this restaurant’s average year (<b>{fmt_lbs(avg_val)}</b>), {HIGHLIGHT_YEAR} was "
            f"<b>{fmt_lbs(delta_abs)} {delta_word}</b> average. "
            f"The highest recorded year was <b>{esc(best_year)}</b> with <b>{fmt_lbs(best_val)}</b>."
        )

    # Colors: highlight 2025
    colors = [COLOR_2025 if str(y) == HIGHLIGHT_YEAR else COLOR_DEFAULT for y in years]

    # ------------------------------------------------------------------
    # FIX: Prevent “fat/wide” charts for 1–2 year restaurants
    #   - fixed bar width
    #   - padded category x-range for 1–2 points
    # ------------------------------------------------------------------
    n_years = len(years)
    if n_years == 1:
        x_range = [-0.6, 0.6]   # centered single bar with padding
    elif n_years == 2:
        x_range = [-0.8, 1.8]   # avoid bars expanding to fill width
    else:
        x_range = None

    fig = go.Figure(
        data=[
            go.Bar(
                x=years,
                y=weights,
                width=BAR_WIDTH,  # <<< prevents fat bars
                marker=dict(color=colors),
                hovertemplate="<b>Year</b>: %{x}<br><b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
            )
        ]
    )

    fig.update_layout(
        title=f"{r} — Yearly Totals",
        xaxis=dict(
            title="Year",
            type="category",
            range=x_range,   # <<< pads the x-axis for 1–2 categories
        ),
        yaxis=dict(
            title="Pounds diverted (Shell Only Weight)",
            rangemode="tozero"
        ),
        bargap=BARGAP,           # <<< consistent bar spacing
        margin=dict(l=60, r=24, t=50, b=42),
        height=CHART_HEIGHT,
    )

    chart_div = fig.to_html(include_plotlyjs=False, full_html=False, config={"displaylogo": False})

    anchor = slugify(r)
    badge = f'<span class="badge badge-yes">{HIGHLIGHT_YEAR} active</span>'

    toc_items.append(f'<li><a href="#{anchor}">{esc(r)}</a> {badge}</li>')

    sections_html.append(
        textwrap.dedent(
            f"""
            <section class="report-card" id="{anchor}">
              <div class="report-head">
                <h2>{esc(r)}</h2>
                <div class="report-meta">
                  <span class="pill pill-2025">{HIGHLIGHT_YEAR}: <b>{fmt_lbs(w_2025)}</b></span>
                  <span class="pill">All years: <b>{fmt_lbs(total_all)}</b></span>
                </div>
              </div>
              <p class="summary">{summary}</p>
              <div class="chart-wrap">{chart_div}</div>
              <div class="backtotop"><a href="#top">Back to top</a></div>
            </section>
            """
        ).strip()
    )

# ------------------------------------------------------------------
# Assemble one HTML document
# ------------------------------------------------------------------
toc_html = "\n".join(toc_items)
body_sections = "\n\n".join(sections_html)

html_doc = textwrap.dedent(
    f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Active Restaurants</title>

  <!-- Load Plotly ONCE -->
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>

  <style>
    :root {{
      --bg: #ffffff;
      --muted: #6b7280;
      --card: #f8fafc;
      --border: #e5e7eb;
      --accent: {COLOR_2025};
    }}

    body {{
      margin: 0;
      font-family: Arial, Helvetica, sans-serif;
      background: var(--bg);
      color: #111;
    }}

    .container {{
      max-width: 1100px;
      margin: 0 auto;
      padding: 18px 16px 50px;
    }}

    h1 {{
      font-size: 22px;
      margin: 0 0 6px;
    }}

    .subtitle {{
      color: var(--muted);
      font-size: 13px;
      margin: 0 0 12px;
      line-height: 1.35;
    }}

    .toc {{
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 10px 12px;
      margin: 10px 0 14px;
    }}

    .toc h3 {{
      margin: 0 0 8px;
      font-size: 14px;
    }}

    .toc ul {{
      margin: 0;
      padding-left: 18px;
      columns: 2;
      column-gap: 22px;
    }}

    @media (max-width: 900px) {{
      .toc ul {{ columns: 1; }}
    }}

    .toc li {{
      margin: 5px 0;
      break-inside: avoid;
    }}

    .toc a {{
      color: #111;
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .toc a:hover {{
      border-bottom-style: solid;
    }}

    .badge {{
      display: inline-block;
      margin-left: 8px;
      font-size: 11px;
      padding: 2px 8px;
      border-radius: 999px;
      border: 1px solid var(--border);
      vertical-align: middle;
      white-space: nowrap;
    }}
    .badge-yes {{
      background: rgba(37, 99, 235, 0.10);
      border-color: rgba(37, 99, 235, 0.35);
      color: #1d4ed8;
    }}

    .report-card {{
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: {CARD_PADDING};
      margin: {CARD_MARGIN};
      background: #fff;
    }}

    .report-head {{
      display: flex;
      align-items: flex-start;
      justify-content: space-between;
      gap: 10px;
      flex-wrap: wrap;
    }}

    .report-card h2 {{
      font-size: 16px;
      margin: 0;
    }}

    .report-meta {{
      display: flex;
      gap: 6px;
      flex-wrap: wrap;
      justify-content: flex-end;
    }}

    .pill {{
      font-size: 12px;
      color: #111;
      background: var(--card);
      border: 1px solid var(--border);
      padding: 3px 9px;
      border-radius: 999px;
    }}

    .pill-2025 {{
      border-color: rgba(37, 99, 235, 0.45);
      background: rgba(37, 99, 235, 0.08);
    }}

    .summary {{
      margin: 8px 0 8px;
      color: #111;
      line-height: 1.35;
      font-size: 13px;
    }}

    .chart-wrap {{
      margin-top: 4px;
    }}

    .backtotop {{
      margin-top: 6px;
      font-size: 12px;
    }}
    .backtotop a {{
      color: var(--muted);
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .backtotop a:hover {{
      border-bottom-style: solid;
    }}

    .footer-note {{
      margin-top: 14px;
      color: var(--muted);
      font-size: 12px;
      line-height: 1.35;
    }}
  </style>
</head>

<body>
  <div class="container" id="top">
    <h1>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Restaurants Active in {HIGHLIGHT_YEAR}</h1>
    <p class="subtitle">
      Metric: <b>Shell Only Weight (lbs)</b> • Only restaurants with recorded contributions in <b>{HIGHLIGHT_YEAR}</b> are included.
      {HIGHLIGHT_YEAR} is highlighted in <span style="color:var(--accent);font-weight:bold;">blue</span>.
    </p>

    <div class="toc">
      <h3>Restaurant index ({len(restaurants_2025)})</h3>
      <ul>
        {toc_html}
      </ul>
    </div>

    {body_sections}

    <div class="footer-note">
      Note: This report aggregates each restaurant’s contributions by year where Shell Only Weight &gt; 0.
      Source: MOP_mega_with_locations_fixed.csv
    </div>
  </div>
</body>
</html>
"""
).strip()

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(html_doc, encoding="utf-8", errors="replace")
print(f"Saved: {OUT_FILE}")

Restaurants active in 2025: 25
Writing one report to: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Saved: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html


In [9]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import re
import textwrap
import html as html_lib
import tempfile
import os

# PDF (ReportLab)
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak, ListFlowable, ListItem
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
CSV = r"C:\Users\suzan\Downloads\MOP\MOP_mega_with_locations_fixed.csv"
OUT_HTML = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html")
OUT_PDF  = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.pdf")

HIGHLIGHT_YEAR = "2025"
COLOR_DEFAULT = "#9CA3AF"   # gray
COLOR_2025 = "#2563EB"      # highlight color

# Layout tuning (thin cards)
CHART_HEIGHT = 280
CARD_PADDING = "10px 12px"
CARD_MARGIN = "10px 0"

# Plot spacing consistency
BAR_WIDTH = 0.5
BARGAP = 0.35

# PDF layout
PDF_PAGE_SIZE = letter
PDF_LEFT_RIGHT_MARGIN = 0.75 * inch
PDF_TOP_BOTTOM_MARGIN = 0.65 * inch
PDF_IMG_MAX_WIDTH = 7.0 * inch   # fit on letter nicely
PDF_IMG_MAX_HEIGHT = 3.4 * inch  # keep sections compact

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def slugify(s: str, max_len: int = 80) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\s\-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = s.strip("-")
    if not s:
        s = "unknown"
    return s[:max_len]

def to_int_year(y):
    try:
        return int(str(y))
    except Exception:
        return None

def fmt_lbs(x):
    try:
        return f"{float(x):,.0f} lbs"
    except Exception:
        return f"{x} lbs"

def esc(s: str) -> str:
    return html_lib.escape(str(s))

def safe_rest_name(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[^\w\s\-\.&]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s if s else "Unknown"

def has_kaleido() -> bool:
    # Plotly uses kaleido under the hood for static image export
    try:
        import kaleido  # noqa: F401
        return True
    except Exception:
        return False

# ------------------------------------------------------------------
# Load & normalize
# ------------------------------------------------------------------
df = pd.read_csv(CSV)

name_col = (
    "Restaurant"
    if "Restaurant" in df.columns
    else ("Restaurants" if "Restaurants" in df.columns else ("name" if "name" in df.columns else None))
)
if name_col is None:
    raise ValueError("Could not find Restaurant column (Restaurant/Restaurants/name).")

weight_col = next((c for c in df.columns if "shell" in c.lower() and "weight" in c.lower()), None)
if not weight_col:
    raise ValueError("No 'Shell Only Weight' column found (shell + weight).")

# Ensure Year exists
if "Year" not in df.columns:
    date_like = next((c for c in df.columns if "date" in c.lower()), None)
    if not date_like:
        raise ValueError("No Year column and no date-like column found to derive Year.")
    df["Year"] = pd.to_datetime(df[date_like], errors="coerce").dt.year

df["Year"] = df["Year"].astype("Int64").astype(str).replace("<NA>", "Unknown")
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)

# Keep only positive weights for impact reporting
df_pos = df[df[weight_col] > 0].copy()

# Only restaurants active in 2025
df_2025 = df_pos[df_pos["Year"].astype(str) == HIGHLIGHT_YEAR].copy()
restaurants_2025 = sorted(df_2025[name_col].fillna("Unknown").astype(str).unique())

if not restaurants_2025:
    raise ValueError(f"No restaurants with Shell Only Weight > 0 found for {HIGHLIGHT_YEAR}.")

print(f"Restaurants active in {HIGHLIGHT_YEAR}: {len(restaurants_2025)}")
print(f"Writing HTML: {OUT_HTML}")
print(f"Writing PDF : {OUT_PDF}")

kaleido_ok = has_kaleido()
if not kaleido_ok:
    print("\n[PDF charts] Kaleido not found, so PDF will be text-only.")
    print("To embed charts in the PDF, install kaleido:")
    print("  pip install -U kaleido\n")

# ------------------------------------------------------------------
# Build per-restaurant content (store data for BOTH HTML and PDF)
# ------------------------------------------------------------------
toc_items = []
sections_html = []
pdf_sections = []  # list of dicts {name, anchor, summary_html, summary_text, fig, w_2025, total_all, best_year, best_val}

for r_raw in restaurants_2025:
    r = safe_rest_name(r_raw)

    sub = df_pos[df_pos[name_col].astype(str) == str(r_raw)].copy()

    year_totals = (
        sub.groupby("Year", as_index=False)[weight_col]
        .sum()
        .rename(columns={weight_col: "Weight"})
    )

    # Sort years numerically when possible
    year_totals["YearInt"] = year_totals["Year"].apply(to_int_year)
    year_totals = year_totals.sort_values(["YearInt", "Year"], na_position="last")

    years = year_totals["Year"].tolist()
    weights = year_totals["Weight"].tolist()

    total_all = float(year_totals["Weight"].sum())
    w_2025 = float(year_totals.loc[year_totals["Year"] == HIGHLIGHT_YEAR, "Weight"].sum())

    best_idx = int(year_totals["Weight"].idxmax())
    best_year = str(year_totals.loc[best_idx, "Year"])
    best_val = float(year_totals.loc[best_idx, "Weight"])

    # --- Narrative WITHOUT averages ---
    if len(year_totals) == 1 and best_year == HIGHLIGHT_YEAR:
        summary_html = (
            f"In <b>{HIGHLIGHT_YEAR}</b>, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"This restaurant has recorded contributions for <b>1 year</b> in the current dataset."
        )
        summary_text = (
            f"In {HIGHLIGHT_YEAR}, {r} diverted {fmt_lbs(w_2025)} of shells from the waste stream. "
            f"This restaurant has recorded contributions for 1 year in the current dataset."
        )
    else:
        summary_html = (
            f"In <b>{HIGHLIGHT_YEAR}</b>, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"Across all recorded years in this dataset, the total is <b>{fmt_lbs(total_all)}</b>. "
            f"The highest recorded year was <b>{esc(best_year)}</b> with <b>{fmt_lbs(best_val)}</b>."
        )
        summary_text = (
            f"In {HIGHLIGHT_YEAR}, {r} diverted {fmt_lbs(w_2025)} of shells from the waste stream. "
            f"Across all recorded years in this dataset, the total is {fmt_lbs(total_all)}. "
            f"The highest recorded year was {best_year} with {fmt_lbs(best_val)}."
        )

    # Colors: highlight 2025
    colors = [COLOR_2025 if str(y) == HIGHLIGHT_YEAR else COLOR_DEFAULT for y in years]

    # FIX: Prevent “fat/wide” charts for 1–2 year restaurants
    n_years = len(years)
    if n_years == 1:
        x_range = [-0.6, 0.6]
    elif n_years == 2:
        x_range = [-0.8, 1.8]
    else:
        x_range = None

n_years = len(years)

if n_years == 1:
    # --- FORCE numeric axis to prevent fat single bar ---
    x_vals = [0]
    x_labels = years

    fig = go.Figure(
        data=[
            go.Bar(
                x=x_vals,
                y=weights,
                width=0.35,  # physical width now respected
                marker=dict(color=[COLOR_2025]),
                hovertemplate="<b>Year</b>: %{customdata}<br>"
                              "<b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
                customdata=x_labels
            )
        ]
    )

    fig.update_layout(
        xaxis=dict(
            title="Year",
            tickvals=[0],
            ticktext=x_labels,
            range=[-0.6, 0.6],
            fixedrange=True
        ),
        yaxis=dict(
            title="Pounds diverted (Shell Only Weight)",
            rangemode="tozero"
        ),
        bargap=0.4,
        margin=dict(l=60, r=24, t=50, b=42),
        height=CHART_HEIGHT,
    )

elif n_years == 2:
    # --- Categorical is OK for 2 bars with padding ---
    fig = go.Figure(
        data=[
            go.Bar(
                x=years,
                y=weights,
                width=0.5,
                marker=dict(color=colors),
                hovertemplate="<b>Year</b>: %{x}<br>"
                              "<b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
            )
        ]
    )

    fig.update_layout(
        xaxis=dict(
            title="Year",
            type="category",
            range=[-0.8, 1.8]
        ),
        yaxis=dict(
            title="Pounds diverted (Shell Only Weight)",
            rangemode="tozero"
        ),
        bargap=0.35,
        margin=dict(l=60, r=24, t=50, b=42),
        height=CHART_HEIGHT,
    )

else:
    # --- Normal categorical behavior ---
    fig = go.Figure(
        data=[
            go.Bar(
                x=years,
                y=weights,
                width=BAR_WIDTH,
                marker=dict(color=colors),
                hovertemplate="<b>Year</b>: %{x}<br>"
                              "<b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
            )
        ]
    )

    fig.update_layout(
        xaxis=dict(
            title="Year",
            type="category"
        ),
        yaxis=dict(
            title="Pounds diverted (Shell Only Weight)",
            rangemode="tozero"
        ),
        bargap=BARGAP,
        margin=dict(l=60, r=24, t=50, b=42),
        height=CHART_HEIGHT,
    )

    fig.update_layout(
        title=f"{r} — Yearly Totals",
        xaxis=dict(title="Year", type="category", range=x_range),
        yaxis=dict(title="Pounds diverted (Shell Only Weight)", rangemode="tozero"),
        bargap=BARGAP,
        margin=dict(l=60, r=24, t=50, b=42),
        height=CHART_HEIGHT,
    )

    chart_div = fig.to_html(include_plotlyjs=False, full_html=False, config={"displaylogo": False})

    anchor = slugify(r)
    badge = f'<span class="badge badge-yes">{HIGHLIGHT_YEAR} active</span>'

    toc_items.append(f'<li><a href="#{anchor}">{esc(r)}</a> {badge}</li>')

    sections_html.append(
        textwrap.dedent(
            f"""
            <section class="report-card" id="{anchor}">
              <div class="report-head">
                <h2>{esc(r)}</h2>
                <div class="report-meta">
                  <span class="pill pill-2025">{HIGHLIGHT_YEAR}: <b>{fmt_lbs(w_2025)}</b></span>
                  <span class="pill">All years: <b>{fmt_lbs(total_all)}</b></span>
                </div>
              </div>
              <p class="summary">{summary_html}</p>
              <div class="chart-wrap">{chart_div}</div>
              <div class="backtotop"><a href="#top">Back to top</a></div>
            </section>
            """
        ).strip()
    )

    pdf_sections.append(
        dict(
            name=r,
            anchor=anchor,
            summary_html=summary_html,
            summary_text=summary_text,
            fig=fig,
            w_2025=w_2025,
            total_all=total_all,
            best_year=best_year,
            best_val=best_val,
        )
    )

# ------------------------------------------------------------------
# Write ONE HTML document
# ------------------------------------------------------------------
toc_html = "\n".join(toc_items)
body_sections = "\n\n".join(sections_html)

html_doc = textwrap.dedent(
    f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Active Restaurants</title>

  <!-- Load Plotly ONCE -->
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>

  <style>
    :root {{
      --bg: #ffffff;
      --muted: #6b7280;
      --card: #f8fafc;
      --border: #e5e7eb;
      --accent: {COLOR_2025};
    }}

    body {{
      margin: 0;
      font-family: Arial, Helvetica, sans-serif;
      background: var(--bg);
      color: #111;
    }}

    .container {{
      max-width: 1100px;
      margin: 0 auto;
      padding: 18px 16px 50px;
    }}

    h1 {{
      font-size: 22px;
      margin: 0 0 6px;
    }}

    .subtitle {{
      color: var(--muted);
      font-size: 13px;
      margin: 0 0 12px;
      line-height: 1.35;
    }}

    .toc {{
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 10px 12px;
      margin: 10px 0 14px;
    }}

    .toc h3 {{
      margin: 0 0 8px;
      font-size: 14px;
    }}

    .toc ul {{
      margin: 0;
      padding-left: 18px;
      columns: 2;
      column-gap: 22px;
    }}

    @media (max-width: 900px) {{
      .toc ul {{ columns: 1; }}
    }}

    .toc li {{
      margin: 5px 0;
      break-inside: avoid;
    }}

    .toc a {{
      color: #111;
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .toc a:hover {{
      border-bottom-style: solid;
    }}

    .badge {{
      display: inline-block;
      margin-left: 8px;
      font-size: 11px;
      padding: 2px 8px;
      border-radius: 999px;
      border: 1px solid var(--border);
      vertical-align: middle;
      white-space: nowrap;
    }}
    .badge-yes {{
      background: rgba(37, 99, 235, 0.10);
      border-color: rgba(37, 99, 235, 0.35);
      color: #1d4ed8;
    }}

    .report-card {{
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: {CARD_PADDING};
      margin: {CARD_MARGIN};
      background: #fff;
    }}

    .report-head {{
      display: flex;
      align-items: flex-start;
      justify-content: space-between;
      gap: 10px;
      flex-wrap: wrap;
    }}

    .report-card h2 {{
      font-size: 16px;
      margin: 0;
    }}

    .report-meta {{
      display: flex;
      gap: 6px;
      flex-wrap: wrap;
      justify-content: flex-end;
    }}

    .pill {{
      font-size: 12px;
      color: #111;
      background: var(--card);
      border: 1px solid var(--border);
      padding: 3px 9px;
      border-radius: 999px;
    }}

    .pill-2025 {{
      border-color: rgba(37, 99, 235, 0.45);
      background: rgba(37, 99, 235, 0.08);
    }}

    .summary {{
      margin: 8px 0 8px;
      color: #111;
      line-height: 1.35;
      font-size: 13px;
    }}

    .chart-wrap {{
      margin-top: 4px;
    }}

    .backtotop {{
      margin-top: 6px;
      font-size: 12px;
    }}
    .backtotop a {{
      color: var(--muted);
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .backtotop a:hover {{
      border-bottom-style: solid;
    }}

    .footer-note {{
      margin-top: 14px;
      color: var(--muted);
      font-size: 12px;
      line-height: 1.35;
    }}
  </style>
</head>

<body>
  <div class="container" id="top">
    <h1>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Restaurants Active in {HIGHLIGHT_YEAR}</h1>
    <p class="subtitle">
      Metric: <b>Shell Only Weight (lbs)</b> • Only restaurants with recorded contributions in <b>{HIGHLIGHT_YEAR}</b> are included.
      {HIGHLIGHT_YEAR} is highlighted in <span style="color:var(--accent);font-weight:bold;">blue</span>.
    </p>

    <div class="toc">
      <h3>Restaurant index ({len(restaurants_2025)})</h3>
      <ul>
        {toc_html}
      </ul>
    </div>

    {body_sections}

    <div class="footer-note">
      Note: This report aggregates each restaurant’s contributions by year where Shell Only Weight &gt; 0.
      Source: MOP_mega_with_locations_fixed.csv
    </div>
  </div>
</body>
</html>
"""
).strip()

OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
OUT_HTML.write_text(html_doc, encoding="utf-8", errors="replace")
print(f"Saved HTML: {OUT_HTML}")

# ------------------------------------------------------------------
# Write ONE PDF document (text + charts if kaleido available)
# ------------------------------------------------------------------
styles = getSampleStyleSheet()
title_style = styles["Title"]
h_style = ParagraphStyle(
    "H",
    parent=styles["Heading2"],
    spaceAfter=6,
    spaceBefore=8,
)
p_style = ParagraphStyle(
    "P",
    parent=styles["BodyText"],
    fontSize=10,
    leading=13,
    spaceAfter=8,
)

small_style = ParagraphStyle(
    "Small",
    parent=styles["BodyText"],
    fontSize=9,
    leading=12,
    textColor="#444444",
)

doc = SimpleDocTemplate(
    str(OUT_PDF),
    pagesize=PDF_PAGE_SIZE,
    leftMargin=PDF_LEFT_RIGHT_MARGIN,
    rightMargin=PDF_LEFT_RIGHT_MARGIN,
    topMargin=PDF_TOP_BOTTOM_MARGIN,
    bottomMargin=PDF_TOP_BOTTOM_MARGIN,
)

story = []
story.append(Paragraph(f"{HIGHLIGHT_YEAR} Shell Recycling Impact Reports", title_style))
story.append(Paragraph(f"Restaurants active in {HIGHLIGHT_YEAR} • Metric: Shell Only Weight (lbs)", small_style))
story.append(Spacer(1, 10))

# Simple TOC-style list (clickable links in PDF are possible, but this is clean + reliable)
toc_list = []
for s in pdf_sections:
    toc_list.append(ListItem(Paragraph(esc(s["name"]), p_style), leftIndent=12))
story.append(Paragraph("Restaurant index", styles["Heading3"]))
story.append(ListFlowable(toc_list, bulletType="bullet", leftIndent=18))
story.append(PageBreak())

with tempfile.TemporaryDirectory() as tmpdir:
    # Render each restaurant section
    for i, s in enumerate(pdf_sections, start=1):
        story.append(Paragraph(s["name"], h_style))

        # A short top-line "pills" equivalent
        line = f"{HIGHLIGHT_YEAR}: <b>{fmt_lbs(s['w_2025'])}</b> &nbsp;&nbsp;|&nbsp;&nbsp; All years: <b>{fmt_lbs(s['total_all'])}</b>"
        story.append(Paragraph(line, p_style))
        story.append(Paragraph(s["summary_text"], p_style))

        # Chart image (if available)
        if kaleido_ok:
            png_path = os.path.join(tmpdir, f"{i:03d}_{slugify(s['name'])}.png")
            try:
                # Higher scale = crisper PDF image
                s["fig"].write_image(png_path, format="png", scale=2)
                img = Image(png_path)

                # constrain image size
                img._restrictSize(PDF_IMG_MAX_WIDTH, PDF_IMG_MAX_HEIGHT)
                story.append(img)
                story.append(Spacer(1, 10))
            except Exception as e:
                story.append(Paragraph(f"(Chart unavailable for PDF: {esc(e)})", small_style))
                story.append(Spacer(1, 8))
        else:
            story.append(Paragraph("(Charts omitted in PDF — install kaleido to embed them.)", small_style))
            story.append(Spacer(1, 8))

        # Page breaks to keep it readable
        if i != len(pdf_sections):
            story.append(PageBreak())

doc.build(story)
print(f"Saved PDF : {OUT_PDF}")


Restaurants active in 2025: 25
Writing HTML: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Writing PDF : C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.pdf
Saved HTML: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Saved PDF : C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.pdf


In [12]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import re
import textwrap
import html as html_lib
import tempfile
import os

# PDF (ReportLab)
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak, ListFlowable, ListItem
)
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
CSV = r"C:\Users\suzan\Downloads\MOP\MOP_mega_with_locations_fixed.csv"
OUT_HTML = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html")
OUT_PDF  = Path(r"C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.pdf")

HIGHLIGHT_YEAR = "2025"
COLOR_DEFAULT = "#9CA3AF"   # gray
COLOR_2025 = "#2563EB"      # highlight color

# Layout tuning (thin cards)
CHART_HEIGHT = 280
CARD_PADDING = "10px 12px"
CARD_MARGIN = "10px 0"

# Plot spacing consistency
BAR_WIDTH = 0.5
BARGAP = 0.35

# PDF layout
PDF_PAGE_SIZE = letter
PDF_LEFT_RIGHT_MARGIN = 0.75 * inch
PDF_TOP_BOTTOM_MARGIN = 0.65 * inch
PDF_IMG_MAX_WIDTH = 7.0 * inch
PDF_IMG_MAX_HEIGHT = 3.4 * inch

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def slugify(s: str, max_len: int = 80) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\s\-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = s.strip("-")
    return (s or "unknown")[:max_len]

def to_int_year(y):
    try:
        return int(str(y))
    except Exception:
        return None

def fmt_lbs(x):
    try:
        return f"{float(x):,.0f} lbs"
    except Exception:
        return f"{x} lbs"

def esc(s: str) -> str:
    return html_lib.escape(str(s))

def safe_rest_name(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[^\w\s\-\.&]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s if s else "Unknown"

def has_kaleido() -> bool:
    try:
        import kaleido  # noqa: F401
        return True
    except Exception:
        return False

# ------------------------------------------------------------------
# Load & normalize
# ------------------------------------------------------------------
df = pd.read_csv(CSV)

name_col = (
    "Restaurant"
    if "Restaurant" in df.columns
    else ("Restaurants" if "Restaurants" in df.columns else ("name" if "name" in df.columns else None))
)
if name_col is None:
    raise ValueError("Could not find Restaurant column (Restaurant/Restaurants/name).")

weight_col = next((c for c in df.columns if "shell" in c.lower() and "weight" in c.lower()), None)
if not weight_col:
    raise ValueError("No 'Shell Only Weight' column found (shell + weight).")

# Ensure Year exists
if "Year" not in df.columns:
    date_like = next((c for c in df.columns if "date" in c.lower()), None)
    if not date_like:
        raise ValueError("No Year column and no date-like column found to derive Year.")
    df["Year"] = pd.to_datetime(df[date_like], errors="coerce").dt.year

df["Year"] = df["Year"].astype("Int64").astype(str).replace("<NA>", "Unknown")
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)

# Keep only positive weights for impact reporting
df_pos = df[df[weight_col] > 0].copy()

# Only restaurants active in 2025
df_2025 = df_pos[df_pos["Year"].astype(str) == HIGHLIGHT_YEAR].copy()
restaurants_2025 = sorted(df_2025[name_col].fillna("Unknown").astype(str).unique())
if not restaurants_2025:
    raise ValueError(f"No restaurants with Shell Only Weight > 0 found for {HIGHLIGHT_YEAR}.")

print(f"Restaurants active in {HIGHLIGHT_YEAR}: {len(restaurants_2025)}")
print(f"Writing HTML: {OUT_HTML}")
print(f"Writing PDF : {OUT_PDF}")

kaleido_ok = has_kaleido()
if not kaleido_ok:
    print("\n[PDF charts] Kaleido not found, so PDF will be text-only.")
    print("To embed charts in the PDF, install kaleido:")
    print("  pip install -U kaleido\n")

# ------------------------------------------------------------------
# Build per-restaurant content (store data for BOTH HTML and PDF)
# ------------------------------------------------------------------
toc_items = []
sections_html = []
pdf_sections = []

for r_raw in restaurants_2025:
    r = safe_rest_name(r_raw)

    sub = df_pos[df_pos[name_col].astype(str) == str(r_raw)].copy()

    year_totals = (
        sub.groupby("Year", as_index=False)[weight_col]
        .sum()
        .rename(columns={weight_col: "Weight"})
    )

    year_totals["YearInt"] = year_totals["Year"].apply(to_int_year)
    year_totals = year_totals.sort_values(["YearInt", "Year"], na_position="last")

    years = year_totals["Year"].tolist()
    weights = year_totals["Weight"].tolist()

    total_all = float(year_totals["Weight"].sum())
    w_2025 = float(year_totals.loc[year_totals["Year"] == HIGHLIGHT_YEAR, "Weight"].sum())

    best_idx = int(year_totals["Weight"].idxmax())
    best_year = str(year_totals.loc[best_idx, "Year"])
    best_val = float(year_totals.loc[best_idx, "Weight"])

    # --- Narrative WITHOUT averages ---
    if len(year_totals) == 1 and best_year == HIGHLIGHT_YEAR:
        summary_html = (
            f"In <b>{HIGHLIGHT_YEAR}</b>, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"This restaurant has recorded contributions for <b>1 year</b> in the current dataset."
        )
        summary_text = (
            f"In {HIGHLIGHT_YEAR}, {r} diverted {fmt_lbs(w_2025)} of shells from the waste stream. "
            f"This restaurant has recorded contributions for 1 year in the current dataset."
        )
    else:
        summary_html = (
            f"In <b>{HIGHLIGHT_YEAR}</b>, <b>{esc(r)}</b> diverted <b>{fmt_lbs(w_2025)}</b> of shells from the waste stream. "
            f"Across all recorded years in this dataset, the total is <b>{fmt_lbs(total_all)}</b>. "
            f"The highest recorded year was <b>{esc(best_year)}</b> with <b>{fmt_lbs(best_val)}</b>."
        )
        summary_text = (
            f"In {HIGHLIGHT_YEAR}, {r} diverted {fmt_lbs(w_2025)} of shells from the waste stream. "
            f"Across all recorded years in this dataset, the total is {fmt_lbs(total_all)}. "
            f"The highest recorded year was {best_year} with {fmt_lbs(best_val)}."
        )

    # Colors: highlight 2025
    colors = [COLOR_2025 if str(y) == HIGHLIGHT_YEAR else COLOR_DEFAULT for y in years]

    # ------------------------------------------------------------------
    # FIX: Prevent “fat” single-bar charts
    #   - 1 year: force numeric axis so width is physically respected
    #   - 2 years: categorical is OK with padding
    #   - 3+ years: normal categorical
    # ------------------------------------------------------------------
    n_years = len(years)

    if n_years == 1:
        x_vals = [0]
        x_labels = years  # e.g. ["2025"]

        fig = go.Figure(
            data=[
                go.Bar(
                    x=x_vals,
                    y=weights,
                    width=0.35,
                    marker=dict(color=[COLOR_2025]),
                    customdata=x_labels,
                    hovertemplate="<b>Year</b>: %{customdata}<br>"
                                  "<b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
                )
            ]
        )

        fig.update_layout(
            title=f"{r} — Yearly Totals",
            xaxis=dict(
                title="Year",
                tickvals=[0],
                ticktext=x_labels,
                range=[-0.6, 0.6],
                fixedrange=True,
            ),
            yaxis=dict(title="Pounds diverted (Shell Only Weight)", rangemode="tozero"),
            bargap=0.45,
            margin=dict(l=60, r=24, t=50, b=42),
            height=CHART_HEIGHT,
        )

    elif n_years == 2:
        fig = go.Figure(
            data=[
                go.Bar(
                    x=years,
                    y=weights,
                    width=0.5,
                    marker=dict(color=colors),
                    hovertemplate="<b>Year</b>: %{x}<br>"
                                  "<b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
                )
            ]
        )

        fig.update_layout(
            title=f"{r} — Yearly Totals",
            xaxis=dict(title="Year", type="category", range=[-0.8, 1.8]),
            yaxis=dict(title="Pounds diverted (Shell Only Weight)", rangemode="tozero"),
            bargap=0.35,
            margin=dict(l=60, r=24, t=50, b=42),
            height=CHART_HEIGHT,
        )

    else:
        fig = go.Figure(
            data=[
                go.Bar(
                    x=years,
                    y=weights,
                    width=BAR_WIDTH,
                    marker=dict(color=colors),
                    hovertemplate="<b>Year</b>: %{x}<br>"
                                  "<b>Shell Only Weight</b>: %{y:,.0f} lbs<extra></extra>",
                )
            ]
        )

        fig.update_layout(
            title=f"{r} — Yearly Totals",
            xaxis=dict(title="Year", type="category"),
            yaxis=dict(title="Pounds diverted (Shell Only Weight)", rangemode="tozero"),
            bargap=BARGAP,
            margin=dict(l=60, r=24, t=50, b=42),
            height=CHART_HEIGHT,
        )

    chart_div = fig.to_html(include_plotlyjs=False, full_html=False, config={"displaylogo": False})

    anchor = slugify(r)
    badge = f'<span class="badge badge-yes">{HIGHLIGHT_YEAR} active</span>'

    toc_items.append(f'<li><a href="#{anchor}">{esc(r)}</a> {badge}</li>')

    sections_html.append(
        textwrap.dedent(
            f"""
            <section class="report-card" id="{anchor}">
              <div class="report-head">
                <h2>{esc(r)}</h2>
                <div class="report-meta">
                  <span class="pill pill-2025">{HIGHLIGHT_YEAR}: <b>{fmt_lbs(w_2025)}</b></span>
                  <span class="pill">All years: <b>{fmt_lbs(total_all)}</b></span>
                </div>
              </div>
              <p class="summary">{summary_html}</p>
              <div class="chart-wrap">{chart_div}</div>
              <div class="backtotop"><a href="#top">Back to top</a></div>
            </section>
            """
        ).strip()
    )

    pdf_sections.append(
        dict(
            name=r,
            anchor=anchor,
            summary_text=summary_text,
            fig=fig,
            w_2025=w_2025,
            total_all=total_all,
        )
    )

# ------------------------------------------------------------------
# Write ONE HTML document
# ------------------------------------------------------------------
toc_html = "\n".join(toc_items)
body_sections = "\n\n".join(sections_html)

html_doc = textwrap.dedent(
    f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Active Restaurants</title>

  <!-- Load Plotly ONCE -->
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>

  <style>
    :root {{
      --bg: #ffffff;
      --muted: #6b7280;
      --card: #f8fafc;
      --border: #e5e7eb;
      --accent: {COLOR_2025};
    }}

    body {{
      margin: 0;
      font-family: Arial, Helvetica, sans-serif;
      background: var(--bg);
      color: #111;
    }}

    .container {{
      max-width: 1100px;
      margin: 0 auto;
      padding: 18px 16px 50px;
    }}

    h1 {{
      font-size: 22px;
      margin: 0 0 6px;
    }}

    .subtitle {{
      color: var(--muted);
      font-size: 13px;
      margin: 0 0 12px;
      line-height: 1.35;
    }}

    .toc {{
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 10px 12px;
      margin: 10px 0 14px;
    }}

    .toc h3 {{
      margin: 0 0 8px;
      font-size: 14px;
    }}

    .toc ul {{
      margin: 0;
      padding-left: 18px;
      columns: 2;
      column-gap: 22px;
    }}

    @media (max-width: 900px) {{
      .toc ul {{ columns: 1; }}
    }}

    .toc li {{
      margin: 5px 0;
      break-inside: avoid;
    }}

    .toc a {{
      color: #111;
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .toc a:hover {{
      border-bottom-style: solid;
    }}

    .badge {{
      display: inline-block;
      margin-left: 8px;
      font-size: 11px;
      padding: 2px 8px;
      border-radius: 999px;
      border: 1px solid var(--border);
      vertical-align: middle;
      white-space: nowrap;
    }}
    .badge-yes {{
      background: rgba(37, 99, 235, 0.10);
      border-color: rgba(37, 99, 235, 0.35);
      color: #1d4ed8;
    }}

    .report-card {{
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: {CARD_PADDING};
      margin: {CARD_MARGIN};
      background: #fff;
    }}

    .report-head {{
      display: flex;
      align-items: flex-start;
      justify-content: space-between;
      gap: 10px;
      flex-wrap: wrap;
    }}

    .report-card h2 {{
      font-size: 16px;
      margin: 0;
    }}

    .report-meta {{
      display: flex;
      gap: 6px;
      flex-wrap: wrap;
      justify-content: flex-end;
    }}

    .pill {{
      font-size: 12px;
      color: #111;
      background: var(--card);
      border: 1px solid var(--border);
      padding: 3px 9px;
      border-radius: 999px;
    }}

    .pill-2025 {{
      border-color: rgba(37, 99, 235, 0.45);
      background: rgba(37, 99, 235, 0.08);
    }}

    .summary {{
      margin: 8px 0 8px;
      color: #111;
      line-height: 1.35;
      font-size: 13px;
    }}

    .chart-wrap {{
      margin-top: 4px;
    }}

    .backtotop {{
      margin-top: 6px;
      font-size: 12px;
    }}
    .backtotop a {{
      color: var(--muted);
      text-decoration: none;
      border-bottom: 1px dotted #bbb;
    }}
    .backtotop a:hover {{
      border-bottom-style: solid;
    }}

    .footer-note {{
      margin-top: 14px;
      color: var(--muted);
      font-size: 12px;
      line-height: 1.35;
    }}
  </style>
</head>

<body>
  <div class="container" id="top">
    <h1>{HIGHLIGHT_YEAR} Shell Recycling Impact Reports — Restaurants Active in {HIGHLIGHT_YEAR}</h1>
    <p class="subtitle">
      Metric: <b>Shell Only Weight (lbs)</b> • Only restaurants with recorded contributions in <b>{HIGHLIGHT_YEAR}</b> are included.
      {HIGHLIGHT_YEAR} is highlighted in <span style="color:var(--accent);font-weight:bold;">blue</span>.
    </p>

    <div class="toc">
      <h3>Restaurant index ({len(restaurants_2025)})</h3>
      <ul>
        {toc_html}
      </ul>
    </div>

    {body_sections}

    <div class="footer-note">
      Note: This report aggregates each restaurant’s contributions by year where Shell Only Weight &gt; 0.
      Source: MOP_mega_with_locations_fixed.csv
    </div>
  </div>
</body>
</html>
"""
).strip()

OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
OUT_HTML.write_text(html_doc, encoding="utf-8", errors="replace")
print(f"Saved HTML: {OUT_HTML}")

# ------------------------------------------------------------------
# Write ONE PDF document (text + charts if kaleido available)
# ------------------------------------------------------------------
styles = getSampleStyleSheet()
title_style = styles["Title"]
h_style = ParagraphStyle("H", parent=styles["Heading2"], spaceAfter=6, spaceBefore=8)
p_style = ParagraphStyle("P", parent=styles["BodyText"], fontSize=10, leading=13, spaceAfter=8)
small_style = ParagraphStyle("Small", parent=styles["BodyText"], fontSize=9, leading=12, textColor="#444444")

doc = SimpleDocTemplate(
    str(OUT_PDF),
    pagesize=PDF_PAGE_SIZE,
    leftMargin=PDF_LEFT_RIGHT_MARGIN,
    rightMargin=PDF_LEFT_RIGHT_MARGIN,
    topMargin=PDF_TOP_BOTTOM_MARGIN,
    bottomMargin=PDF_TOP_BOTTOM_MARGIN,
)

story = []
story.append(Paragraph(f"{HIGHLIGHT_YEAR} Shell Recycling Impact Reports", title_style))
story.append(Paragraph(f"Restaurants active in {HIGHLIGHT_YEAR} • Metric: Shell Only Weight (lbs)", small_style))
story.append(Spacer(1, 10))

toc_list = [ListItem(Paragraph(esc(s["name"]), p_style), leftIndent=12) for s in pdf_sections]
story.append(Paragraph("Restaurant index", styles["Heading3"]))
story.append(ListFlowable(toc_list, bulletType="bullet", leftIndent=18))
story.append(PageBreak())

with tempfile.TemporaryDirectory() as tmpdir:
    for i, s in enumerate(pdf_sections, start=1):
        story.append(Paragraph(s["name"], h_style))
        line = f"{HIGHLIGHT_YEAR}: <b>{fmt_lbs(s['w_2025'])}</b> &nbsp;&nbsp;|&nbsp;&nbsp; All years: <b>{fmt_lbs(s['total_all'])}</b>"
        story.append(Paragraph(line, p_style))
        story.append(Paragraph(s["summary_text"], p_style))

        if kaleido_ok:
            png_path = os.path.join(tmpdir, f"{i:03d}_{slugify(s['name'])}.png")
            try:
                s["fig"].write_image(png_path, format="png", scale=2)
                img = Image(png_path)
                img._restrictSize(PDF_IMG_MAX_WIDTH, PDF_IMG_MAX_HEIGHT)
                story.append(img)
                story.append(Spacer(1, 10))
            except Exception as e:
                story.append(Paragraph(f"(Chart unavailable for PDF: {esc(e)})", small_style))
                story.append(Spacer(1, 8))
        else:
            story.append(Paragraph("(Charts omitted in PDF — install kaleido to embed them.)", small_style))
            story.append(Spacer(1, 8))

        if i != len(pdf_sections):
            story.append(PageBreak())

doc.build(story)
print(f"Saved PDF : {OUT_PDF}")


Restaurants active in 2025: 26
Writing HTML: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Writing PDF : C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.pdf
Saved HTML: C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.html
Saved PDF : C:\Users\suzan\Downloads\MOP\docs\impact_reports_2025_all.pdf
